# Excel reports through ZEMI Arsenal and JSON Schema

`MarkItDown` converts three Excel reports to Markdown; llama.cpp applies the JSON Schema generated by Pydantic during inference, and Pydantic validates the resulting JSON.

In [ ]:
model_name = "ling30_tiny"

In [1]:
from pathlib import Path

while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

WindowsPath('c:/Users/Axoman/Documents/ZEMI/zemi_component_template')

In [9]:
import zemi
from zemi.arsenal import ArsenalSession

arsenal = ArsenalSession()
zemi.arsenal.begin(arsenal, stop_before_begin=True, llama_router_mode=True)
model = arsenal.model(model_name)
assistant = model.assistants["assistant"]
client = assistant.clients.openai.client.with_options(timeout=300.0, max_retries=0)


══════════════════════════════════════════════════════════════════════════════
ZEMI Arsenal · STOP ARSENAL
Llama servers in configuration: 1
══════════════════════════════════════════════════════════════════════════════
[1/1] curated_router · 127.0.0.1:8080
    ✓ found and stopped
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal stopped
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Arsenal · ARSENAL READY · ROUTER MODE
Llama servers in configuration: 1
══════════════════════════════════════════════════════════════════════════════
Download and startup are deferred until the model is first accessed.
Example: arsenal.llamas["curated_router"].models["qwen35_4b"]
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Arsenal · 

## Pydantic schema

In [3]:
from pydantic import BaseModel, ConfigDict, Field

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class Transaction(StrictModel):
    date: str = Field(description="Date in YYYY-MM-DD format")
    article: int
    cost: float

class Report(StrictModel):
    source_file: str
    city: str
    export_date: str = Field(description="Date in YYYY-MM-DD format")
    manager: str
    transactions: list[Transaction]

class Reports(StrictModel):
    reports: list[Report]


## Excel to Markdown

In [4]:
from markitdown import MarkItDown
from zemi import env

data_dir = env.path.comp.root / "data/case01"
excel_files = [data_dir / f"Report {number}.xlsx" for number in range(1, 4)]
if not all(path.is_file() for path in excel_files):
    excel_files = [data_dir / f"Отчет {number}.xlsx" for number in range(1, 4)]
converter = MarkItDown(enable_plugins=False)
excel_context = "\n\n".join(
    f"# File: {path.name}\n\n{converter.convert(path).text_content.strip()}"
    for path in excel_files
)
[(path.name, path.stat().st_size) for path in excel_files]

[('Отчет 1.xlsx', 10365), ('Отчет 2.xlsx', 10291), ('Отчет 3.xlsx', 10530)]

## JSON Schema-constrained extraction

In [5]:
import json

task = """
Extract every report: source file, city/branch, export date, manager, and
transactions (date, article, cost). Articles are integers. Omit Total
rows, invent nothing, write
all dates as YYYY-MM-DD, and return compact JSON without indentation.
""".strip()

response = client.chat.completions.create(
    model=assistant.clients.model,  # ling-3.0-tiny
    messages=[
        {"role": "system", "content": "Convert Excel reports into strictly structured JSON."},
        {"role": "user", "content": f"{task}\n\n{excel_context}"},
    ],
    temperature=0.0,
    max_tokens=2800,
    extra_body={"json_schema": Reports.model_json_schema()},
)
result = Reports.model_validate_json(response.choices[0].message.content)
print(json.dumps(result.model_dump(mode="json"), ensure_ascii=False, indent=2))

{
  "reports": [
    {
      "source_file": "Отчет 1.xlsx",
      "city": "Москва",
      "export_date": "2026-04-30",
      "manager": "Иванов И.И.",
      "transactions": [
        {
          "date": "2026-03-14",
          "article": 15,
          "cost": 83204.0
        },
        {
          "date": "2026-03-24",
          "article": 61,
          "cost": 53307.0
        },
        {
          "date": "2026-03-09",
          "article": 56,
          "cost": 67750.0
        },
        {
          "date": "2026-05-17",
          "article": 65,
          "cost": 21224.0
        },
        {
          "date": "2026-04-21",
          "article": 78,
          "cost": 65206.0
        },
        {
          "date": "2026-03-27",
          "article": 61,
          "cost": 40616.0
        },
        {
          "date": "2026-02-14",
          "article": 14,
          "cost": 33626.0
        }
      ]
    },
    {
      "source_file": "Отчет 2.xlsx",
      "city": "Санкт-Петербург",
      "

In [6]:
result

Reports(reports=[Report(source_file='Отчет 1.xlsx', city='Москва', export_date='2026-04-30', manager='Иванов И.И.', transactions=[Transaction(date='2026-03-14', article=15, cost=83204.0), Transaction(date='2026-03-24', article=61, cost=53307.0), Transaction(date='2026-03-09', article=56, cost=67750.0), Transaction(date='2026-05-17', article=65, cost=21224.0), Transaction(date='2026-04-21', article=78, cost=65206.0), Transaction(date='2026-03-27', article=61, cost=40616.0), Transaction(date='2026-02-14', article=14, cost=33626.0)]), Report(source_file='Отчет 2.xlsx', city='Санкт-Петербург', export_date='2026-04-30', manager='Петров П.П.', transactions=[Transaction(date='2026-01-27', article=21, cost=3515.0), Transaction(date='2026-05-11', article=48, cost=1969.0), Transaction(date='2026-02-18', article=33, cost=2964.0), Transaction(date='2026-05-02', article=16, cost=5848.0)]), Report(source_file='Отчет 3.xlsx', city='Самара', export_date='2026-04-30', manager='Сидоров С.С.', transactio

## Stop Arsenal

Run this cell when inference is complete.

In [7]:
zemi.arsenal.end(arsenal, stop_after_end=True)


══════════════════════════════════════════════════════════════════════════════
ZEMI Arsenal · STOP ARSENAL
Llama servers in configuration: 1
══════════════════════════════════════════════════════════════════════════════
[1/1] curated_router · 127.0.0.1:8080
    ✓ stopped · PID 15000
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal stopped
══════════════════════════════════════════════════════════════════════════════


## Output parameters

Publish structured JSON results for the ZEMI job report. The custom MIME output is the marker; no cell tag is required.

In [ ]:
from zemi.playbook import output_params

reports = result.model_dump(mode="json")["reports"]
output_params({
    "report_count": len(reports),
    "reports": reports,
})